# 12 - Prediction, Inference & Batch Scoring

## 1. Production Inference Engine
In this notebook, we demonstrate the production inference workflow using `LoanPredictor`.
Features:
- Single customer prediction with full propensity segmentation.
- High-throughput batch scoring on marketing lists.
- Boundary condition and edge case testing.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.predict import LoanPredictor

predictor = LoanPredictor()
print("Predictor successfully initialized.")
print("Model name:", predictor.metadata.get("best_model_name"))
print("Calibrated decision threshold:", predictor.decision_threshold)

2026-10-26 17:55:00 [INFO] model_predict (predict.py:42): Loading model artifact from D:\Personal lone\artifacts\models\bank_personal_loan_model.joblib


2026-10-26 17:55:00 [INFO] model_predict (predict.py:48): Predictor initialized with model: CatBoost (threshold: 0.634)


Predictor successfully initialized.
Model name: CatBoost
Calibrated decision threshold: 0.6336363636363637


## 2. Single Customer Scenario Testing

In [2]:
# Scenario A: High propensity affluent customer (CD account holder, high income, graduate)
customer_a = {
    "Age": 42,
    "Experience": 18,
    "Income": 165,
    "ZIP Code": 94720,
    "Family": 3,
    "CCAvg": 5.2,
    "Education": 3,
    "Mortgage": 0,
    "Securities Account": 0,
    "CD Account": 1,
    "Online": 1,
    "CreditCard": 0,
}

# Scenario B: Low propensity young depositor (low income, undergraduate, no auxiliary accounts)
customer_b = {
    "Age": 24,
    "Experience": -1,  # Edge case: negative experience to verify automated cleaning
    "Income": 28,
    "ZIP Code": 90089,
    "Family": 1,
    "CCAvg": 0.5,
    "Education": 1,
    "Mortgage": 0,
    "Securities Account": 0,
    "CD Account": 0,
    "Online": 0,
    "CreditCard": 0,
}

res_a = predictor.predict_single(customer_a)
res_b = predictor.predict_single(customer_b)

print("--- Scenario A (Affluent Customer) ---")
print(res_a)
print("\n--- Scenario B (Young Low-Income Depositor) ---")
print(res_b)

--- Scenario A (Affluent Customer) ---
{'prediction': 1, 'loan_accepted': True, 'acceptance_probability': 1.0, 'propensity_segment': 'High Propensity', 'decision_threshold_used': 0.6336363636363637}

--- Scenario B (Young Low-Income Depositor) ---
{'prediction': 0, 'loan_accepted': False, 'acceptance_probability': 0.0, 'propensity_segment': 'Low Propensity', 'decision_threshold_used': 0.6336363636363637}


## 3. High-Throughput Batch Inference

In [3]:
test_df = pd.read_csv(project_root / "data" / "processed" / "test.csv")
customers_batch = test_df.drop(columns=["Personal Loan"]).to_dict(orient="records")[:20]

batch_predictions = predictor.predict_batch(customers_batch)
results_df = pd.DataFrame(batch_predictions)
display(results_df.head(10))
print("\nPropensity Segment Distribution in Batch:")
print(results_df["propensity_segment"].value_counts())

,probability,prediction,loan_accepted,propensity_segment
0,0.0000,0,False,Low Propensity
1,0.9999,1,True,High Propensity
2,0.0000,0,False,Low Propensity
3,0.0000,0,False,Low Propensity
4,0.0000,0,False,Low Propensity
5,0.0000,0,False,Low Propensity
6,0.0000,0,False,Low Propensity
7,0.0000,0,False,Low Propensity
8,0.0000,0,False,Low Propensity
9,0.0007,0,False,Low Propensity



Propensity Segment Distribution in Batch:
propensity_segment
Low Propensity     19
High Propensity     1
Name: count, dtype: int64
